# scFoundation C2 训练分析

在正式结果目录运行。GEARS 的训练与验证曲线来自官方入口的原始文本日志，回执记录预算和更新次数。

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt

root = Path.cwd()
result = json.loads((root / "result.json").read_text())
fit = json.loads((root / "fit_receipt.json").read_text())
assert result["status"] == "evaluated" and result["checkpoint_sha256"] == fit["checkpoint_sha256"]
log = (root / "fit.log").read_text()
number = r"([-+\d.]+(?:[eE][-+]?\d+)?)"
steps = [(int(step), float(loss.rstrip("."))) for step, loss in re.findall(
    r"Epoch \d+ Step (\d+) Train Loss: " + number, log
)]
overall = re.findall(r"Train Overall MSE: " + number + r"\s+Validation Overall MSE: " + number, log)
de = re.findall(r"Train Top 20 DE MSE: " + number + r"\s+Validation Top 20 DE MSE: " + number, log)
assert steps and len(overall) == len(de) == fit["epochs"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), layout="constrained")
axes[0].plot(*zip(*steps), color="#285d9f")
axes[0].set(xlabel="Microbatch", ylabel="GEARS training loss")
for ax, values, title in zip(axes[1:], [overall, de], ["Overall", "Source-selected DE20"], strict=True):
    train, val = [float(x.rstrip(".")) for x in values[-1]]
    bars = ax.bar(["Train", "Validation"], [train, val], color=["#285d9f", "#c66d24"])
    ax.bar_label(bars, fmt="%.4f")
    ax.set(title=title, ylabel="MSE")
fig.suptitle(f'{fit["fit_microbatches"]} microbatches · {fit["optimizer_updates"]} updates')
fig.savefig(root / "training-curves.png", dpi=170)
plt.show()